In [1]:
import numpy as np
import sklearn as sk
import pandas as pd
import data_management as dm
import feature_builders as fb
import tree_models as tm
import model_wrappers as wp
from graph_helpers import plot_var_vs_loss

In [2]:
TICKERS = ["AAPL", "JPM", "XOM", "KO"]

WEIGHTS = {
    "AAPL": 0.25,
    "JPM": 0.25,
    "XOM": 0.25,
    "KO": 0.25,
}

df = pd.read_csv(
    "./data_general/4_ticker_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)


tickers_data = df[TICKERS].pct_change().dropna()



In [3]:
calendar_last_180_days = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start="2024-06-03",   # Pazartesi, ~6 ay geriye (180 takvim günü civarı)
    train_end="2024-11-29",     # Cuma, Kasım'ın son iş günü
    forecast_start="2024-12-02",  # Pazartesi
    forecast_end="2024-12-31",
)

[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 4 weekday days had no data and were skipped (likely market holidays): 2024-06-19, 2024-07-04, 2024-09-02, 2024-11-28
[INFO] train set: 126 bars, 4 weekday days skipped
[WARNING] [forecast] 1 weekday day had no data and were skipped (likely market holidays): 2024-12-25
[INFO] forecast set: 21 bars, 1 weekday day skipped


In [4]:
training_feature_long, forecast_feature_long = fb.build_long_features(calendar_last_180_days)

[INFO] long train: 504 rows -> 424 after dropna (80 dropped, 15.9%)
[INFO] long forecast: 84 rows -> 84 after dropna (0 dropped, 0.0%)


In [ ]:
WINDOW_SIZE= 50
VAR_CONFIDENCE = 0.95
model = tm.xgboost_train(calendar_last_180_days, training_feature_long, WINDOW_SIZE)

[INFO] usable_train_dates: 96/105 candidate dates survived burn-in (min_periods=29, train_data starts 2024-06-03, burn-in cutoff 2024-07-16, 9 candidate date(s) dropped).


In [19]:
var_df, residualbank, ewmabank = dm.multi_day_var_from_long_format_model(
    pooled_model=model,
    calendar=calendar_last_180_days,
    long_forecast_feature=forecast_feature_long,
    multi_day_var_calculation_window_size=360,  # int: use at most the last N residuals; None: use all residuals available
    var_confidence=VAR_CONFIDENCE,                      # e.g. 0.95 -> takes the 5% lower quantile of the residuals
    initial_residual_bank=model.residual_bank_in_sample,
    initial_ewma_variance_bank=model.ewma_variance_bank_in_sample,
    run_key=None,                                       # optional: key in calendar.var_prediction_runs (default: model.model_name)
    multi_day_var_min_periods=30,                       # optional: min prior residuals a day needs to get a VaR (else ValueError)
)

In [20]:
plot_var_vs_loss(var_df, residualbank)